In [1]:
import os
import json
import sys
import argparse
import datetime
import pandas as pd
import time
import traceback
from trade_core import InitCore
# from exchange_plugin.common_info import InitCommonInfo
# from monitor_engine.kimp_core_monitor import InitKimpCoreMonitor
# from etc.db_handler.create_schema_tables import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.redis_connector.redis_connector import InitRedis
import _pickle as pickle

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [2]:
# redis_cli = InitRedis()

In [3]:
logging_dir = "/home/trade_core/"
with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)
proc_n = 1
# node = config['node']
node = 'trade_core_dev'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [4]:
core = InitCore(logging_dir, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, db_dict)

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
2024-02-28 08:20:25,844 - trade_core - INFO - InitCore|enabled_market_code_services: ['UPBIT_SPOT/KRW:BINANCE_USD_M/USDT', 'BITHUMB_SPOT/KRW:OKX_USD_M/USDT']


InitDBClient|SCHEMA: trade_core already exists.
InitDBClient|TABLE: trade_config already exists.
InitDBClient|TABLE: trade already exists.
InitDBClient|TABLE: repeat_trade already exists.
InitDBClient|TABLE: exchange_api_key already exists.


2024-02-28 08:20:26,104 - trade_core - INFO - InitCore|server_check_status_list has been initiated.
2024-02-28 08:20:26,106 - trade_core - INFO - InitCore|InitCore initiated with proc_n=1
2024-02-28 08:20:26,825 - okx_plug - INFO - okx_plug_logger started.
2024-02-28 08:20:27,209 - upbit_plug - INFO - upbit_plug_logger started.
2024-02-28 08:20:27,226 - update_dollar - INFO - fetch_dollar|Dollar price (1336.0 KRW) has been updated.
2024-02-28 08:20:27,374 - binance_plug - INFO - binance_plug_logger started.
2024-02-28 08:20:27,376 - bithumb_plug - INFO - bithumb_plug_logger started.
2024-02-28 08:20:27,377 - bybit_plug - INFO - bybit_plug_logger started.
2024-02-28 08:20:27,378 - trade_core - INFO - InitCore|update_binance_spot_ticker_df thread has started.
2024-02-28 08:20:27,382 - trade_core - INFO - InitCore|update_okx_spot_info_df thread has started.
2024-02-28 08:20:27,387 - trade_core - INFO - InitCore|update_binance_usd_m_info_df thread has started.
2024-02-28 08:20:27,391 - tra

In [ ]:
core.server_check_status_list

In [ ]:
target_market_code = "BITHUMB_SPOT/KRW"
origin_market_code = "OKX_USD_M/USDT"
core.get_premium_df(target_market_code, origin_market_code)

In [ ]:
import time
import traceback
import pandas as pd
from etc.db_handler.postgres_client import InitDBClient as InitPostgresDBClient
from loggers.logger import KimpBotLogger
from threading import Thread
from multiprocessing import Process, Manager
from threading import Thread
from psycopg2 import extras
from etc.acw_api import AcwApi

class InitTrigger:
    def __init__(self, admin_telegram_id, node, server_check_status_list, get_premium_df, enabled_markets, register_monitor_msg, remote_redis_client, db_dict, mongo_client, postgres_client, logging_dir):
        self.admin_telegram_id = admin_telegram_id
        self.node = node
        self.acw_api = AcwApi(prod=False)
        self.server_check_status_list = server_check_status_list
        self.get_premium_df = get_premium_df
        self.enabled_markets = enabled_markets
        self.register_monitor_msg = register_monitor_msg
        self.remote_redis_client = remote_redis_client
        self.db_dict = db_dict
        self.mongo_client = mongo_client
        self.postgres_client = postgres_client
        self.logging_dir = logging_dir
        self.logger = KimpBotLogger("trigger", logging_dir).logger
        self.manager = Manager()
        self.exchange_config_df_dict = self.manager.dict()
        self.exchange_config_df_dict_initiated = False
        self.free_trade_df_dict = self.manager.dict()
        self.trade_df_dict = self.manager.dict()
        self.load_exchange_config_thread = Thread(target=self.load_exchange_config, daemon=True)
        self.load_exchange_config_thread.start()

        self.trade_proc_dict = {}
        self.alarm_proc_dict = {}
        for each_market_combi_code in self.enabled_markets:        
            self.trade_proc_dict[each_market_combi_code] = Process(target=self.start_trigger_loop, args=(each_market_combi_code, self.trade_df_dict, False, 'trade', 2), daemon=True)
            self.trade_proc_dict[each_market_combi_code].start()
            self.alarm_proc_dict[each_market_combi_code] = Process(target=self.start_trigger_loop, args=(each_market_combi_code, self.free_trade_df_dict, True, 'trade', 2), daemon=True)
            self.alarm_proc_dict[each_market_combi_code].start()
        # while self.user_info_dict_initiated is False or self.exchange_config_df_dict_initiated is False or [self.trade_df_dict.get(each_market_combi_code) for each_market_combi_code in self.enabled_markets].count(None) != 0:
        #     time.sleep(0.2)
        # while self.user_info_dict_initiated is False:
        #     time.sleep(0.2)
        self.logger.info("trade_config_df_dict, trade_df_dict have been initialized")
        time.sleep(20)

    def is_table_empty(self, conn, table_name):
        with conn.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cur.fetchone()[0]
            return count == 0

    def get_column_names(self, conn, table_name):
        with conn.cursor() as cur:
            cur.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_schema = 'public' AND table_name = %s
                ORDER BY ordinal_position;
            """, (table_name,))
            return [row[0] for row in cur.fetchall()]

    def load_exchange_config(self, table_name='trade_config'):
        try:
            # First check whether the table is empty
            conn = self.postgres_client.pool.getconn()
            curr = conn.cursor(cursor_factory=extras.RealDictCursor)
            if self.is_table_empty(conn, table_name):
                # # Get the column names
                # column_names = self.get_column_names(conn, table_name)
                # # Create empty dataframe
                # self.exchange_config_df = pd.DataFrame(columns=column_names)
                pass
            else:
                curr.execute(f"SELECT * FROM {table_name}")
                exchange_config_df = pd.DataFrame(curr.fetchall())
                target_market_code_unique = exchange_config_df['target_market_code'].unique()
                origin_market_code_unique = exchange_config_df['origin_market_code'].unique()
                for target_market_code in target_market_code_unique:
                    for origin_market_code in origin_market_code_unique:
                        market_combi_code = f"{target_market_code}:{origin_market_code}"
                        self.exchange_config_df_dict[market_combi_code] = exchange_config_df[(exchange_config_df['target_market_code'] == target_market_code) & (exchange_config_df['origin_market_code'] == origin_market_code)]
            self.postgres_client.pool.putconn(conn)
            if self.exchange_config_df_dict_initiated is False:
                self.exchange_config_df_dict_initiated = True
        except Exception as e:
            # rollback the transaction if any error while inserting
            self.postgres_client.pool.putconn(conn, close=True)
            title = f"Error in load_exchange_config"
            full_content = f"{title}:\n{traceback.format_exc()}"
            self.logger.error(full_content)
            self.acw_api.create_message(self.admin_telegram_id, title, self.node, "monitor", full_content, code=None)
            time.sleep(5)

    def load_exchange_config_loop(self, table_name='trade_config', loop_interval_secs=1):
        while True:
            self.load_exchange_config(table_name)
            time.sleep(loop_interval_secs)

    def load_trade_df(self, conn, curr, market_combi_code, trade_df_dict, free_user, table_name='trade'):
        target_market_code, origin_market_code = market_combi_code.split(':')
        try:
            # First check whether the table is empty
            if self.is_table_empty(conn, table_name):
                # Get the column names
                column_names = self.get_column_names(conn, table_name)
                # Create empty dataframe
                trade_df_dict[market_combi_code] = pd.DataFrame(columns=column_names)
            else:
                if free_user:
                    sql = """
                    SELECT trade.*, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code, trade_config.send_times, trade_config.send_term
                    FROM trade
                    JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.target_market_code=%s AND trade_config.origin_market_code=%s AND trade_config.service_datetime_end <= %s"""
                else:
                    sql = """
                    SELECT trade.*, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code, trade_config.send_times, trade_config.send_term,
                    trade_config.target_market_cross, trade_config.target_market_leverage, trade_config.origin_market_cross, trade_config.origin_market_leverage, trade_config.target_market_margin_call, trade_config.origin_market_margin_call,
                    trade_config.target_market_safe_reverse, trade_config.origin_market_safe_reverse, trade_config.target_market_risk_threshold_p, trade_config.origin_market_risk_threshold_p, trade_config.repeat_limit_p,
                    trade_config.repeat_limit_direction, trade_config.repeat_num_limit
                    FROM trade
                    JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.target_market_code=%s AND trade_config.origin_market_code=%s AND trade_config.service_datetime_end > %s"""
                val = (target_market_code, origin_market_code, pd.Timestamp.now())
                curr.execute(sql, val)
                trade_df = pd.DataFrame(curr.fetchall())
                trade_df_dict[market_combi_code] = trade_df
                return trade_df
        except Exception as e:
            title = f"Error in load_trade_df"
            full_content = f"Error in load_trade_df: {e}\n{traceback.format_exc()}"
            self.logger.error(full_content)
            self.acw_api.create_message(self.admin_telegram_id, title, self.node, "monitor", full_content, code=None)
        
    def start_trigger_loop(self, market_combi_code, trade_df_dict, free_user=True, table_name='trade', loop_interval_secs=5):
        self.logger.info(f"start_trigger_loop: {market_combi_code} | free_user: {free_user} | table_name: {table_name} | loop_interval_secs: {loop_interval_secs}")
        postgres_client = InitPostgresDBClient(**{**self.db_dict, 'database': 'trade_core'})
        conn = postgres_client.pool.getconn()
        curr = conn.cursor(cursor_factory=extras.RealDictCursor)
        self.logger.info(f"postgres client has been initiated for {market_combi_code}")
        while True:
            try:
                trade_df = self.load_trade_df(conn, curr, market_combi_code, trade_df_dict, free_user, table_name)
                # Loading data done
                if len(trade_df) == 0 or trade_df is None:
                    time.sleep(loop_interval_secs)
                    continue

                premium_df = self.get_premium_df(*market_combi_code.split(':'))
                merged_df = trade_df.merge(premium_df, on='base_asset')
                merged_df['SL_premium_value'] = merged_df.apply(lambda x: x['SL_premium'] if x['usdt_conversion'] == False else (1+x['SL_premium']/100)*x['dollar'], axis=1)
                merged_df['LS_premium_value'] = merged_df.apply(lambda x: x['LS_premium'] if x['usdt_conversion'] == False else (1+x['LS_premium']/100)*x['dollar'], axis=1)

                # switch None: 최초, 0: 하향돌파 시, 1: 상향돌파 시
                # auto_trade_switch 0: 진입대기, -1: 탈출대기, 1:탈출완료, 2:탈출에러
                # case 1. switch None or False(0), High 돌파
                high_break_trade_df = (merged_df[((merged_df['trigger_switch'].isnull())|(merged_df['trigger_switch']==False))
                            &(merged_df['SL_premium_value']>=merged_df['high'])&(merged_df['trade_switch']!=3)])

                # case 2. switch None or True(1), Low 돌파
                low_break_trade_df = (merged_df[((merged_df['trigger_switch'].isnull())|(merged_df['trigger_switch']==True))
                            &(merged_df['LS_premium_value']<=merged_df['low'])&(merged_df['trade_switch']!=3)])

                # trade_swtich == 0: 진입대기, -1: 탈출대기, 1:탈출완료, 2:탈출에러, 3:거래 진행 중
                if len(high_break_trade_df) != 0:
                    high_break_trigger_uuid_list = high_break_trade_df['uuid'].to_list()
                    # UPDATE database
                    # conn = postgres_client.pool.getconn()
                    # curr = conn.cursor()
                    curr.execute(f"UPDATE trade SET trigger_switch = 1 WHERE uuid IN %s", (tuple(high_break_trigger_uuid_list),))
                    conn.commit()

                    # postgres_client.pool.putconn(conn)
                    self.high_break(high_break_trade_df, free_user)

                if len(low_break_trade_df) != 0:
                    low_break_trigger_uuid_list = low_break_trade_df['uuid'].to_list()
                    # UPDATE database
                    # conn = postgres_client.pool.getconn()
                    # curr = conn.cursor()
                    curr.execute(f"UPDATE trade SET trigger_switch = 0 WHERE uuid IN %s", (tuple(low_break_trigger_uuid_list),))
                    conn.commit()
                    # postgres_client.pool.putconn(conn)
                    self.low_break(low_break_trade_df, free_user)

                time.sleep(loop_interval_secs)
            except Exception as e:
                title = f"Error in start_trigger_loop"
                full_content = f"{title}: {e}\n{traceback.format_exc()}"
                self.logger.error(full_content)
                # rollback the transaction if any error while inserting
                postgres_client.pool.putconn(conn, close=True)
                self.acw_api.create_message(self.admin_telegram_id, title, self.node, "monitor", full_content, code=None)
                time.sleep(10)

    def high_break(self, high_break_trade_df, free_user):
        for row_tup in high_break_trade_df.iterrows():
            def row_thread(row_tup):
                row = row_tup[1]

                ls_premium = row['LS_premium']
                ls_premium_value = row['LS_premium_value']
                sl_premium = row['SL_premium']
                sl_premium_value = row['SL_premium_value']

                msg_title = f"{row['base_asset']}/{row['quote_asset']} 프리미엄 상향돌파"
                msg_content = f"{row['target_market_code']}:{row['origin_market_code']}\n"
                msg_content += f"현재 SL:{round(sl_premium_value, 3)}, 설정된 탈출값: {row['high']}\n"
                if pd.isnull(row['tp']):
                    current_price = (row['ap'] + row['bp'])/2
                else:
                    current_price = row['tp']
                msg_content += f"현재가격: {current_price}({round(row['scr'],2)}%)"
                msg_full = f"{msg_title}\n{msg_content}"
                self.acw_api.create_message_thread(row['telegram_id'], msg_title, self.node, 'info', msg_full, send_times=row['send_times'], send_term=row['send_term'])
            row_thread = Thread(target=row_thread, args=(row_tup,), daemon=True)
            row_thread.start()

    def low_break(self, low_break_trade_df, free_user):
        for row_tup in low_break_trade_df.iterrows():
            def row_thread(row_tup):
                row = row_tup[1]
                base_asset = row['base_asset']
                quote_asset = row['quote_asset']                

                ls_premium = row['LS_premium']
                ls_premium_value = row['LS_premium_value']
                sl_premium = row['SL_premium']
                sl_premium_value = row['SL_premium_value']

                msg_title = f"{base_asset}/{quote_asset} 프리미엄 하향돌파"
                msg_content = f"{row['target_market_code']}:{row['origin_market_code']}\n"
                msg_content += f"현재 LS:{round(ls_premium_value, 3)}, 설정된 진입값: {row['low']}\n"
                if pd.isnull(row['tp']):
                    current_price = (row['ap'] + row['bp'])/2
                else:
                    current_price = row['tp']
                msg_content += f"현재가격: {current_price}({round(row['scr'],2)}%)"
                msg_full = f"{msg_title}\n{msg_content}"
                self.acw_api.create_message_thread(row['telegram_id'], msg_title, self.node, 'info', msg_full, send_times=row['send_times'], send_term=row['send_term'])
            row_thread = Thread(target=row_thread, args=(row_tup,), daemon=True)
            row_thread.start()

In [ ]:
trigger = InitTrigger(admin_id, node, core.server_check_status_list, core.get_premium_df, core.enabled_market_code_services, register_monitor_msg, core.remote_redis_client, core.db_dict, core.mongo_client, core.postgres_client, None)

In [ ]:
from exchange_plugin.binance_plug import UserBinanceAdaptor

In [6]:
core.trigger.trade_df_dict.get('UPBIT_SPOT/KRW:BINANCE_USD_M/USDT')

""


In [7]:
core.trigger.free_trade_df_dict.get('UPBIT_SPOT/KRW:BINANCE_USD_M/USDT')

,id,uuid,trade_config_uuid,registered_datetime,last_updated_datetime,base_asset,usdt_conversion,low,high,trigger_switch,...,exit_origin_market_order_id,status,remark,trade_config,service_datetime_end,telegram_id,target_market_code,origin_market_code,send_times,send_term
0,127,ac3eed6d-e1e3-4dfe-a18a-6710e4020b92,b4cc70d0-6a8e-4605-b0ee-bde6be062cae,2024-02-22 06:20:16.487929,2024-02-22 06:20:16.487932,BTC,True,-1376.000,1378.000,1,...,None,None,a,"(52,b4cc70d0-6a8e-4605-b0ee-bde6be062cae,6e4fa...",2024-02-14 07:10:15.731422,1722781031,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
1,114,adbdd01f-240a-4f45-886b-66037ec182e1,cb9ff909-7a01-490d-aedd-5f5f8c35c607,2024-02-16 05:45:35.591171,2024-02-16 05:45:35.591179,BTC,True,1380.000,1390.000,0,...,None,None,None,"(55,cb9ff909-7a01-490d-aedd-5f5f8c35c607,ab2a3...",2024-02-16 05:37:59.076599,1390695186,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
2,132,ff823e38-e703-4d78-ab32-707da3ed34a3,5026038d-00d0-4feb-af08-b7d27f4407fb,2024-02-23 10:43:39.089834,2024-02-23 10:43:39.089843,BTC,True,1398.000,1400.000,0,...,None,None,None,"(62,5026038d-00d0-4feb-af08-b7d27f4407fb,7a867...",2024-02-23 10:43:38.963407,1722781031,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,3,1
3,126,a0a05577-3ddd-44d8-981b-06903fc47484,5184e17a-ecc3-4cef-a63e-ac96e58ddb7e,2024-02-20 03:43:59.699631,2024-02-20 03:43:59.699637,BTC,True,1384.000,1388.500,0,...,None,None,None,"(54,5184e17a-ecc3-4cef-a63e-ac96e58ddb7e,b0275...",2024-02-16 04:15:14.841154,1731633310,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
4,19,e9275836-4489-4211-bd91-40c932783ec3,00dc872d-b31d-412d-94e5-9de9026f4a48,2024-02-05 03:04:33.917853,2024-02-05 03:04:33.917859,BTC,False,1.000,2.000,1,...,None,None,None,"(44,00dc872d-b31d-412d-94e5-9de9026f4a48,87c2c...",2024-02-05 03:03:35.349363,1722781031,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
5,20,4230280a-8e45-411b-9d16-5ce42f8a1776,00dc872d-b31d-412d-94e5-9de9026f4a48,2024-02-05 03:20:00.728998,2024-02-05 03:20:00.729002,BTC,False,1.000,2.000,1,...,None,None,None,"(44,00dc872d-b31d-412d-94e5-9de9026f4a48,87c2c...",2024-02-05 03:03:35.349363,1722781031,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
6,123,e62a3c34-85ea-4940-8c07-041bc027e060,57f5df1d-95f9-49eb-b7e4-c15042c9b59f,2024-02-19 09:15:59.814960,2024-02-19 09:15:59.814965,BTC,False,0.000,2.000,1,...,None,None,None,"(2,57f5df1d-95f9-49eb-b7e4-c15042c9b59f,d1b3f6...",2024-02-01 09:14:47.168000,1390695186,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
7,125,e6db17b0-b602-47c5-971c-74d96d1c5f7a,5184e17a-ecc3-4cef-a63e-ac96e58ddb7e,2024-02-20 03:09:44.411698,2024-02-20 03:09:44.411704,VET,False,3.200,3.400,1,...,None,None,None,"(54,5184e17a-ecc3-4cef-a63e-ac96e58ddb7e,b0275...",2024-02-16 04:15:14.841154,1731633310,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
8,133,aadb347b-1ad1-46f3-a876-bd56a5c31075,57f5df1d-95f9-49eb-b7e4-c15042c9b59f,2024-02-28 07:22:42.251150,2024-02-28 07:22:42.251159,BTC,False,6.000,7.000,0,...,None,None,None,"(2,57f5df1d-95f9-49eb-b7e4-c15042c9b59f,d1b3f6...",2024-02-01 09:14:47.168000,1390695186,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1
9,115,d7b7322e-0c2a-4476-89d2-b22661c585aa,cb9ff909-7a01-490d-aedd-5f5f8c35c607,2024-02-16 05:47:42.035462,2024-02-16 05:47:42.035465,BTC,False,9.000,10.000,0,...,None,None,None,"(55,cb9ff909-7a01-490d-aedd-5f5f8c35c607,ab2a3...",2024-02-16 05:37:59.076599,1390695186,UPBIT_SPOT/KRW,BINANCE_USD_M/USDT,1,1


2024-02-28 07:24:50,997 - update_dollar - INFO - fetch_dollar|Dollar price (1334.5 KRW) has been updated.


In [ ]:
user_binance_adaptor = UserBinanceAdaptor(admin_id, node, db_dict, core.trigger.free_trade_df_dict, 'UPBIT_SPOT/KRW:BINANCE_USD_M/USDT')

2024-02-28 07:25:09,034 - user_binance_plug - INFO - user_binance_plug_logger|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT started.


InitDBClient|SCHEMA: trade_core already exists.


In [ ]:
user_binance_adaptor.start_user_socket_stream(market_type="USD_M")

In [ ]:
user_binance_adaptor.user_stream_monitoring_list

In [ ]:
conn = trigger.postgres_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
SELECT *
FROM user_info
INNER JOIN exchange_config ON user_info.user_uuid = exchange_config.user_uuid;
"""

sql = """
SELECT * FROM user_info
"""


start = time.time()
curr.execute(sql)
result = curr.fetchall()
print(time.time() - start)
# temp_df = pd.DataFrame(result)

trigger.postgres_client.pool.putconn(conn)


In [ ]:
result[0]

In [ ]:
# Add addcoin manually

from uuid import uuid4

conn = trigger.postgres_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
INSERT INTO trade (user_uuid, last_updated_datetime, uuid, base_asset, usdt_conversion, target_market_code, origin_market_code, low, high) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);
"""
user_uuid = "test_uuid"
last_updated_datetime = datetime.datetime.utcnow()
uuid = uuid4().hex
base_asset = "BTC"
usdt_conversion = False
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
low = 5
high = 6

curr.execute(sql, (user_uuid, last_updated_datetime, uuid, base_asset, usdt_conversion, target_market_code, origin_market_code, low, high))
conn.commit()

trigger.postgres_client.pool.putconn(conn)

In [ ]:
conn.rollback()
trigger.postgres_client.pool.putconn(conn)